In [12]:
# Import necessary libraries
import numpy as np
import pandas as pd
from pathlib import Path
import os

# Reproducibility
np.random.seed(42)

In [13]:
# Pipe configurations
pipe_length   = 50.0
pipe_diameter = 0.2
pipe_radius   = 0.1

nodes_per_timestep = 146
timesteps          = 700

# Blockage-specific decay lengths much longer than leak
# Back pressure travels many pipe diameters upstream
BLOCKAGE_UPSTREAM_DECAY   = 15 * pipe_diameter   # 3.0 m
BLOCKAGE_DOWNSTREAM_DECAY = 10 * pipe_diameter   # 2.0 m

baseline = {
    'pressure':             19657.0,
    'pressure-coefficient': 32600.0,
    'density':              2700.0,
    'velocity-magnitude':   2.479,
    'x-velocity':           2.478,
    'y-velocity':           0.0017,
    'z-velocity':          -0.0013,
    'turb-kinetic-energy':  0.0363,
    'turb-diss-rate':       24.4,
    'wall-shear':           10.4,
    'tailings-vof':         0.400,
}

noise = {
    'pressure':             500.0,
    'pressure-coefficient': 800.0,
    'density':              0.0,
    'velocity-magnitude':   0.05,
    'x-velocity':           0.05,
    'y-velocity':           0.005,
    'z-velocity':           0.005,
    'turb-kinetic-energy':  0.002,
    'turb-diss-rate':       1.5,
    'wall-shear':           0.5,
    'tailings-vof':         0.005,
}

# Output directory
base_dir = os.path.join(os.path.dirname(os.getcwd()), "data", "synthetic")

# Simulation variation parameters
scenarios = [
    {"id": 1,  "velocity_factor": 1.00, "concentration_factor": 0.95},
    {"id": 2,  "velocity_factor": 1.00, "concentration_factor": 1.05},
    {"id": 3,  "velocity_factor": 0.95, "concentration_factor": 1.00},
    {"id": 4,  "velocity_factor": 1.05, "concentration_factor": 1.00},
    {"id": 5,  "velocity_factor": 0.95, "concentration_factor": 0.95},
    {"id": 6,  "velocity_factor": 0.95, "concentration_factor": 1.05},
    {"id": 7,  "velocity_factor": 1.05, "concentration_factor": 0.95},
    {"id": 8,  "velocity_factor": 1.05, "concentration_factor": 1.05},
    {"id": 9,  "velocity_factor": 0.98, "concentration_factor": 0.98},
    {"id": 10, "velocity_factor": 1.02, "concentration_factor": 1.02},
    {"id": 11, "velocity_factor": 0.97, "concentration_factor": 1.03},
    {"id": 12, "velocity_factor": 1.03, "concentration_factor": 0.97},
    {"id": 13, "velocity_factor": 0.96, "concentration_factor": 1.04},
    {"id": 14, "velocity_factor": 1.04, "concentration_factor": 0.96},
]

# Velocity factor scales
def apply_scaling(base, velocity_factor, concentration_factor):
    scaled = {}

    # Velocity-related scaling
    scaled["velocity_magnitude"] = base["velocity_magnitude"] * velocity_factor
    scaled["x_velocity"]         = base["x_velocity"] * velocity_factor
    scaled["flow_velocity"]      = base["flow_velocity"] * velocity_factor

    # Concentration scaling
    scaled["tailings_vof"] = base["tailings_vof"] * concentration_factor
    scaled["density"]      = 2700  # constant

    # Pressure unchanged
    scaled["pressure"] = base["pressure"]

    # Turbulence scaling
    scaled["tke"] = base["tke"] * (velocity_factor ** 2)
    scaled["tdr"] = base["tdr"] * (velocity_factor ** 3)

    return scaled

# ------ 9. Configuration Summary ------
print("========== CONFIG SUMMARY ==========")
print(f"Pipe Length: {pipe_length}")
print(f"Pipe Diameter: {pipe_diameter}")
print(f"Nodes per Timestep: {nodes_per_timestep}")
print(f"Timesteps: {timesteps}")
print(f"Total Scenarios: {len(scenarios)}")
print(f"Output Directory: {base_dir}")
print("====================================")


========== CONFIG SUMMARY ==========
Pipe Length: 50.0
Pipe Diameter: 0.2
Nodes per Timestep: 146
Timesteps: 700
Total Scenarios: 14
Output Directory: /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/synthetic


In [14]:
# Node co-ordinate generator
def generate_node_coordinates(n_nodes):
    np.random.seed(42)
    node_numbers = np.arange(1, n_nodes + 1)
    x_coords = np.linspace(0, pipe_length, n_nodes)
    angles = np.linspace(0,2*np.pi, n_nodes)
    radii = np.random.uniform(0, pipe_radius, n_nodes)
    y_coords = radii * np.cos(angles)
    z_coords = radii * np.sin(angles)
    return node_numbers, x_coords, y_coords, z_coords

# Function to generate normal pressure profiles
def generate_normal_pressure_profile(x_coords):
    P_inlet = 40000.0
    P_outlet = 0.0

    return P_inlet - (P_inlet - P_outlet) * (x_coords / pipe_length)
